In [1]:
import warnings, time, joblib
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE, BorderlineSMOTE, ADASYN

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.metrics import f1_score, classification_report
from sklearn.base import clone

# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
# from optuna.samplers import TPESampler

pd.set_option('display.max_columns', None)
plt.rcParams.update({'figure.dpi': 100})

In [2]:
import sys
from pathlib import Path

src_path = (Path.cwd().resolve().parents[1] / "src").as_posix()
if src_path not in sys.path:
    sys.path.append(src_path)

import generic_anomaly_pipeline as gap



In [3]:

config = gap.AnomalyConfig(
    train_path=Path("../../data/processed_data/wot/wot_train_ml.parquet"),
    tune_path=Path("../../data/processed_data/wot/wot_val_ml.parquet"),
    test_path=Path("../../data/processed_data/wot/wot_val_ml.parquet"),
    output_model=Path("models/wot_to_wot_anomaly.joblib"),
    output_report=Path("reports/wot_to_wot_anomaly.json"),
    label_col="label",
    normal_label=0,
    threshold_quantile=0.13,
    seed=7524,
    use_custom_stopwords=False,
    tfidf=gap.DEFAULT_TFIDF,
    text_col="clean_message",
)

models = gap.build_detectors(config)


In [4]:
train_df = gap.load_frame(config.train_path, config.text_col, config.label_col)
tune_df = gap.load_frame(config.tune_path, config.text_col, config.label_col)
test_df = gap.load_frame(config.test_path, config.text_col, config.label_col)

In [5]:
vectorizer = TfidfVectorizer(**config.tfidf)
X_train_vec = vectorizer.fit_transform(train_df[config.text_col])

In [6]:
# display the first 5 rows of the vectorized training data as a dense array
print(X_train_vec[:5].todense())

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [7]:
models

{'IsolationForest': IsolationForest(n_estimators=300, n_jobs=-1, random_state=7524),
 'OneClassSVM': OneClassSVM(nu=0.13),
 'SGDOneClassSVM': SGDOneClassSVM(max_iter=2000, nu=0.13, random_state=7524),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.1, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=6, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=300, n_jobs=None,
               num_parallel_tree=None, ...)}